# Comparative Analysis of SAM, MobileSAM and FastSAM with Fine-Tuning of DeepLabV3


1. Prepare selected COCO records and masks.
2. Evaluate prompt-based models with the same box-prompt protocol.
3. Evaluate pretrained DeepLabV3 as the supervised baseline.
4. Fine-tune DeepLabV3 in small trials first, then use the larger final checkpoint.

## 1. Setup, Paths and Run Flags


In [ ]:
# Main paths and switches used to rerun or skip expensive experiment parts.
# Most flags stay False when the saved outputs are already present.

from pathlib import Path
import json
import math
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Find the project folder even if the notebook is opened from another directory.
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'selected').exists() and (candidate / 'results' / 'tables').exists():
            return candidate
    return start

ROOT = find_project_root()
DATA_DIR = ROOT / 'data'
SELECTED_DIR = DATA_DIR / 'selected'
RESULTS_DIR = ROOT / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
OUTPUTS_DIR = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUTS_DIR / 'checkpoints'

# Expected COCO locations if the full dataset is restored.
COCO_DIR = DATA_DIR / 'coco'
COCO_TRAIN_IMAGES = COCO_DIR / 'train2017'
COCO_VAL_IMAGES = COCO_DIR / 'val2017'
COCO_ANN_DIR = COCO_DIR / 'annotations'
COCO_TRAIN_ANN = COCO_ANN_DIR / 'instances_train2017.json'
COCO_VAL_ANN = COCO_ANN_DIR / 'instances_val2017.json'

# Expected model weights if prompt-model evaluation is rerun.
SAM_CHECKPOINT = ROOT / 'models/weights/sam_vit_b_01ec64.pth'
SAM_VIT_H_CHECKPOINT = ROOT / 'models/weights/sam_vit_h_4b8939.pth'
MOBILE_SAM_CHECKPOINT = ROOT / 'models/weights/mobile_sam.pt'
FASTSAM_CHECKPOINT = ROOT / 'models/weights/FastSAM-s.pt'
FINAL_RAW_DEEPLAB_CHECKPOINT = CHECKPOINT_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2.pth'
FINAL_DEEPLAB_CHECKPOINT = CHECKPOINT_DIR / 'deeplabv3_finetuned_model_soup_50k.pth'

# Heavy experiment flags. Set to True to rerun the corresponding experiments and regenerate all artifacts, or False to skip and use existing artifacts if available.
REBUILD_SELECTED_RECORDS = False
REGENERATE_SANITY_FIGURES = False
REGENERATE_VISUAL_COMPARISON = False
# Set these to True only when I want to regenerate that part of the experiment.
RUN_PROMPT_MODEL_EVAL = False
RUN_SAM_VIT_H_EVAL = False
RUN_DEEPLAB_BASELINE_EVAL = False
RUN_DEEPLAB_FINETUNED_EVAL = False
RUN_DEEPLAB_TRAINING = False
# If a trial checkpoint/evaluation already exists, keep it and load the saved result.
RERUN_COMPLETED_DEEPLAB_TRIALS = False
RUN_MODEL_SOUP_SEARCH = False
RUN_FULL_VALIDATION_EVAL = False
REBUILD_SUMMARY_TABLES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')

def read_csv(name):
    return pd.read_csv(TABLES_DIR / name)

# Used by skipped cells to confirm that saved outputs already exist.
def artifact_exists(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0

# Small helper for readable notebook output.
def print_lines(title, lines):
    print(title)
    for line in lines:
        print(f'- {line}')

def yes_no(value):
    return 'available' if value else 'not available in this folder'

train_image_status = (
    'available'
    if COCO_TRAIN_IMAGES.exists()
    else 'not available in this folder; only needed to rerun DeepLabV3 training from scratch'
)

setup_lines = [
    f'COCO train images: {train_image_status}',
    f'COCO validation images: {yes_no(COCO_VAL_IMAGES.exists())}',
    f'COCO train annotations: {yes_no(COCO_TRAIN_ANN.exists())}',
    f'COCO validation annotations: {yes_no(COCO_VAL_ANN.exists())}',
    f'SAM checkpoint: {yes_no(artifact_exists(SAM_CHECKPOINT))}',
    f'MobileSAM checkpoint: {yes_no(artifact_exists(MOBILE_SAM_CHECKPOINT))}',
    f'FastSAM checkpoint: {yes_no(artifact_exists(FASTSAM_CHECKPOINT))}',
    f'Final DeepLabV3 checkpoint: {yes_no(artifact_exists(FINAL_DEEPLAB_CHECKPOINT))}',
]
print_lines('Runtime setup check', setup_lines)

## 2. COCO and Mask Helper Code


In [ ]:
# COCO annotations are stored as JSON, so I build small lookup tables first.
# These helpers are reused for masks, selected objects and evaluation.

def load_coco(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

# Convert COCO lists into dictionaries for quick lookup by id.
def build_coco_indices(coco):
    images = {int(img['id']): img for img in coco.get('images', [])}
    categories = {int(cat['id']): cat for cat in coco.get('categories', [])}
    annotations_by_image = {}
    annotations_by_id = {}
    for ann in coco.get('annotations', []):
        image_id = int(ann['image_id'])
        annotations_by_image.setdefault(image_id, []).append(ann)
        annotations_by_id[int(ann['id'])] = ann
    return {
        'images': images,
        'categories': categories,
        'annotations_by_image': annotations_by_image,
        'annotations_by_id': annotations_by_id,
    }

# SAM-style box prompts use [x1, y1, x2, y2].
def bbox_xywh_to_xyxy(bbox_xywh):
    x, y, w, h = [float(v) for v in bbox_xywh]
    return [x, y, x + w, y + h]

def bbox_to_center_point(bbox_xywh):
    x, y, w, h = [float(v) for v in bbox_xywh]
    return [x + w / 2.0, y + h / 2.0]

def annotation_category_name(annotation, categories):
    return categories[int(annotation['category_id'])]['name']

# COCO segmentations can be polygons or RLE; both become a boolean mask.
def annotation_to_mask(annotation, height, width):
    from pycocotools import mask as mask_utils
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        rles = mask_utils.frPyObjects(segmentation, height, width)
        rle = mask_utils.merge(rles)
    elif isinstance(segmentation, dict) and isinstance(segmentation.get('counts'), list):
        rle = mask_utils.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation
    return mask_utils.decode(rle).astype(bool)

# Pick one clear target object from an image for the selected-object protocol.
def choose_primary_annotation(annotations, categories, allowed_names=None, min_area=300.0):
    candidates = []
    for ann in annotations:
        if ann.get('iscrowd', 0):
            continue
        if float(ann.get('area', 0.0)) < min_area:
            continue
        name = annotation_category_name(ann, categories)
        if allowed_names is not None and name not in allowed_names:
            continue
        candidates.append(ann)
    if not candidates:
        return None
    return max(candidates, key=lambda ann: float(ann.get('area', 0.0)))

# IoU is overlap divided by union, used as the main mask-quality score.
def binary_iou(pred_mask, gt_mask):
    pred = np.asarray(pred_mask).astype(bool)
    gt = np.asarray(gt_mask).astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    if union == 0:
        return 1.0
    return float(intersection / union)

def binary_dice(pred_mask, gt_mask):
    pred = np.asarray(pred_mask).astype(bool)
    gt = np.asarray(gt_mask).astype(bool)
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denom)

VOC_CATEGORIES = [
    '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor'
]
VOC_NAME_TO_INDEX = {name: idx for idx, name in enumerate(VOC_CATEGORIES)}
# Only these COCO classes match the VOC-style DeepLabV3 output head.
COCO_TO_VOC_NAME = {
    'person': 'person', 'bicycle': 'bicycle', 'car': 'car', 'motorcycle': 'motorbike',
    'airplane': 'aeroplane', 'bus': 'bus', 'train': 'train', 'boat': 'boat', 'bird': 'bird',
    'cat': 'cat', 'dog': 'dog', 'horse': 'horse', 'sheep': 'sheep', 'cow': 'cow',
    'bottle': 'bottle', 'chair': 'chair', 'couch': 'sofa', 'potted plant': 'pottedplant',
    'dining table': 'diningtable', 'tv': 'tvmonitor'
}
COCO_TO_VOC_INDEX = {coco_name: VOC_NAME_TO_INDEX[voc_name] for coco_name, voc_name in COCO_TO_VOC_NAME.items()}
VOC_COMPATIBLE_COCO_NAMES = set(COCO_TO_VOC_INDEX)

print(f'VOC-compatible COCO classes used for DeepLabV3: {len(VOC_COMPATIBLE_COCO_NAMES)}')

## 3. Step 1: Build Selected Validation Record Lists


In [ ]:
# Build a selected validation list with one usable target object per image.
# Small or invalid masks are skipped so the IoU comparison is meaningful.

def build_primary_object_records(annotations_json, images_dir, output_json, limit=None, allowed_names=None, min_area=300.0):
    coco = load_coco(annotations_json)
    indices = build_coco_indices(coco)
    image_paths = sorted(Path(images_dir).glob('*.jpg'))
    rows = []
    for image_path in image_paths:
        # COCO image id is encoded in the image filename.
        try:
            image_id = int(image_path.stem)
        except ValueError:
            continue
        image = indices['images'].get(image_id)
        if image is None:
            continue
        annotations = indices['annotations_by_image'].get(image_id, [])
        # Choose one target object for this image.
        target = choose_primary_annotation(annotations, indices['categories'], allowed_names=allowed_names, min_area=min_area)
        if target is None:
            continue
        # Store only the fields needed later for prompts, masks and IoU.
        rows.append({
            'image_id': image_id,
            'file_name': image['file_name'],
            'width': int(image['width']),
            'height': int(image['height']),
            'annotation_id': int(target['id']),
            'category_id': int(target['category_id']),
            'category_name': annotation_category_name(target, indices['categories']),
            'bbox_xywh': [float(v) for v in target['bbox']],
            'area': float(target.get('area', 0.0)),
        })
        if limit is not None and len(rows) >= limit:
            break
    # Save the selected record list so every model uses the same images and objects.
    write_json(output_json, rows)
    return rows

if REBUILD_SELECTED_RECORDS:
    build_primary_object_records(COCO_VAL_ANN, COCO_VAL_IMAGES, SELECTED_DIR / 'val_1000_primary_objects.json', limit=1000)
    build_primary_object_records(COCO_VAL_ANN, COCO_VAL_IMAGES, SELECTED_DIR / 'val_all_primary_objects.json', limit=None)
    record_list_status = 'Selected validation records were rebuilt from COCO annotations.'
else:
    record_list_status = 'Selected validation record JSON files are available in data/selected.'

record_list_names = ['val_1000_primary_objects.json', 'val_all_primary_objects.json']
record_list_counts = {name: len(read_json(SELECTED_DIR / name)) for name in record_list_names if (SELECTED_DIR / name).exists()}
example_val_record = read_json(SELECTED_DIR / 'val_1000_primary_objects.json')[0]

print_lines('Selected validation records step', [
    record_list_status,
    f"val_1000 records used for the complete all-model comparison: {record_list_counts.get('val_1000_primary_objects.json', 0)}",
    f"val_all records used for the robustness check: {record_list_counts.get('val_all_primary_objects.json', 0)}",
    f"Example target object: image {example_val_record['image_id']}, category '{example_val_record['category_name']}', annotation {example_val_record['annotation_id']}",
])

## 4. Step 2: Build DeepLabV3 Training Dataset


In [ ]:
# Build selected COCO train2017 records for DeepLabV3 fine-tuning.
# Only images with VOC-compatible classes are useful for this DeepLabV3 head.

def build_deeplab_training_records(annotations_json, images_dir, output_json, limit=None, seed=42, min_area=300.0):
    rng = random.Random(seed)
    coco = load_coco(annotations_json)
    indices = build_coco_indices(coco)
    rows = []
    for image_id, image in sorted(indices['images'].items()):
        image_path = Path(images_dir) / image['file_name']
        if not image_path.exists():
            continue
        annotations = indices['annotations_by_image'].get(image_id, [])
        # Collect objects that can be mapped to DeepLabV3/VOC classes.
        compatible = []
        for ann in annotations:
            if ann.get('iscrowd', 0):
                continue
            name = annotation_category_name(ann, indices['categories'])
            if name in VOC_COMPATIBLE_COCO_NAMES and float(ann.get('area', 0.0)) >= min_area:
                compatible.append(ann)
        if not compatible:
            continue
        names = sorted({annotation_category_name(ann, indices['categories']) for ann in compatible})
        # One training record can contain several compatible objects/classes.
        rows.append({
            'image_id': int(image_id),
            'file_name': image['file_name'],
            'width': int(image['width']),
            'height': int(image['height']),
            'annotation_ids': [int(ann['id']) for ann in compatible],
            'category_ids': sorted({int(ann['category_id']) for ann in compatible}),
            'category_names': names,
            'object_count': len(compatible),
            'foreground_area': float(sum(float(ann.get('area', 0.0)) for ann in compatible)),
        })
    # Shuffle before limiting so subsets are not ordered by COCO image id.
    rng.shuffle(rows)
    if limit is not None:
        rows = rows[:limit]
    write_json(output_json, rows)
    return rows

if REBUILD_SELECTED_RECORDS:
    build_deeplab_training_records(COCO_TRAIN_ANN, COCO_TRAIN_IMAGES, SELECTED_DIR / 'train_2000_deeplabv3.json', limit=2000)
    build_deeplab_training_records(COCO_TRAIN_ANN, COCO_TRAIN_IMAGES, SELECTED_DIR / 'train_5000_deeplabv3.json', limit=5000)
    build_deeplab_training_records(COCO_TRAIN_ANN, COCO_TRAIN_IMAGES, SELECTED_DIR / 'train_10000_deeplabv3.json', limit=10000)
    build_deeplab_training_records(COCO_TRAIN_ANN, COCO_TRAIN_IMAGES, SELECTED_DIR / 'train_50000_deeplabv3.json', limit=50000)
    train_record_list_status = 'Selected training records were rebuilt from COCO training annotations.'
else:
    train_record_list_status = 'DeepLabV3 selected training record JSON files are available in data/selected.'

train_record_list_names = ['train_2000_deeplabv3.json', 'train_5000_deeplabv3.json', 'train_10000_deeplabv3.json', 'train_50000_deeplabv3.json']
train_counts = {name: len(read_json(SELECTED_DIR / name)) for name in train_record_list_names if (SELECTED_DIR / name).exists()}
example_train_record = read_json(SELECTED_DIR / 'train_50000_deeplabv3.json')[0]

print_lines('DeepLabV3 selected training records step', [
    train_record_list_status,
    f"first fine-tuning subset: {train_counts.get('train_2000_deeplabv3.json', 0)} records",
    f"larger training record lists prepared: {train_counts.get('train_5000_deeplabv3.json', 0)}, {train_counts.get('train_10000_deeplabv3.json', 0)} and {train_counts.get('train_50000_deeplabv3.json', 0)} records",
    f"Example training image {example_train_record['image_id']} contains mapped categories: {', '.join(example_train_record['category_names'])}",
])

## 5. Step 3: Verify Class Mapping and Masks


In [ ]:
# Save the exact COCO-to-DeepLabV3 class mapping used in training and evaluation.

# This table is a quick audit of the class ids used by DeepLabV3.
mapping_check = pd.DataFrame([
    {
        'coco_name': coco_name,
        'voc_label_name': voc_name,
        'deeplab_output_index': VOC_NAME_TO_INDEX[voc_name],
    }
    for coco_name, voc_name in COCO_TO_VOC_NAME.items()
]).sort_values('deeplab_output_index')

mapping_output = TABLES_DIR / 'deeplabv3_class_mapping.csv'
mapping_check.to_csv(mapping_output, index=False)
assert artifact_exists(mapping_output)
print(f'Generated {mapping_output.relative_to(ROOT)} with {len(mapping_check)} mapped classes')

In [ ]:
# Create semantic training masks where pixel values are DeepLabV3 class indices.
# Unsupported COCO objects are marked as 255 so the loss ignores those pixels.

def build_semantic_mask_for_image(annotations, height, width, categories, ignore_index=255, ignore_unmapped_objects=True):
    mask = np.zeros((height, width), dtype=np.uint8)
    # Larger objects are written first, then smaller objects can overwrite their own pixels.
    sorted_annotations = sorted(annotations, key=lambda ann: float(ann.get('area', 0.0)), reverse=True)

    # Mark unsupported classes first so they do not become false background labels.
    if ignore_unmapped_objects:
        for ann in sorted_annotations:
            if ann.get('iscrowd', 0):
                continue
            name = annotation_category_name(ann, categories)
            if name not in COCO_TO_VOC_INDEX:
                ann_mask = annotation_to_mask(ann, height, width)
                mask[ann_mask] = ignore_index

    # Write supported VOC-compatible classes into the semantic mask.
    for ann in sorted_annotations:
        if ann.get('iscrowd', 0):
            continue
        name = annotation_category_name(ann, categories)
        label = COCO_TO_VOC_INDEX.get(name)
        if label is None:
            continue
        ann_mask = annotation_to_mask(ann, height, width)
        mask[ann_mask] = int(label)
    return mask

# Optional sanity figure: check that COCO masks are being read correctly.
if (REGENERATE_SANITY_FIGURES or not artifact_exists(FIGURES_DIR / 'step1_dataset_sample.png')) and COCO_VAL_ANN.exists() and COCO_VAL_IMAGES.exists():
    from PIL import Image
    coco_val = load_coco(COCO_VAL_ANN)
    indices_val = build_coco_indices(coco_val)
    record = read_json(SELECTED_DIR / 'val_1000_primary_objects.json')[0]
    image = np.array(Image.open(COCO_VAL_IMAGES / record['file_name']).convert('RGB'))
    ann = indices_val['annotations_by_id'][int(record['annotation_id'])]
    gt_mask = annotation_to_mask(ann, record['height'], record['width'])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"Image {record['image_id']} | {record['category_name']}")
    axes[0].axis('off')
    axes[1].imshow(gt_mask, cmap='gray')
    axes[1].set_title('COCO target mask')
    axes[1].axis('off')
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / 'step1_dataset_sample.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

sanity_figure = FIGURES_DIR / 'step1_dataset_sample.png'
assert artifact_exists(sanity_figure)
print(f'Sanity figure available: {sanity_figure.relative_to(ROOT)}')

## 6. Step 4: Prompt-Based Model Evaluation Code


In [ ]:
# Prompt-based models receive the selected object bounding box and return one mask.
# I save per-image IoU and runtime so the summary tables can be rebuilt later.

def get_device(prefer_cuda=True):
    try:
        import torch
        if prefer_cuda and torch.cuda.is_available():
            return 'cuda'
    except Exception:
        pass
    return 'cpu'

# Run once for warmup, then measure the actual inference call.
def measure_runtime(fn, warmup_runs=1, timed_runs=1):
    for _ in range(warmup_runs):
        fn()
    times = []
    result = None
    for _ in range(timed_runs):
        start = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - start)
    return result, {'mean_seconds': float(np.mean(times)), 'std_seconds': float(np.std(times))}

# Wrapper for SAM with a box prompt.
class SAMBoxRunner:
    def __init__(self, checkpoint, model_type='vit_b', device=None):
        from segment_anything import SamPredictor, sam_model_registry
        self.device = device or get_device()
        model = sam_model_registry[model_type](checkpoint=str(checkpoint))
        model.to(device=self.device)
        model.eval()
        self.predictor = SamPredictor(model)

    def predict_box(self, image_rgb, box_xyxy):
        self.predictor.set_image(image_rgb)
        masks, scores, _ = self.predictor.predict(
            box=np.array(box_xyxy, dtype=np.float32),
            # Use one mask per prompt so all models produce one prediction per object.
            multimask_output=False,
        )
        return masks[0], float(scores[0])

# MobileSAM uses the same box-prompt idea with a lighter encoder.
class MobileSAMBoxRunner:
    def __init__(self, checkpoint, model_type='vit_t', device=None):
        from mobile_sam import SamPredictor, sam_model_registry
        self.device = device or get_device()
        model = sam_model_registry[model_type](checkpoint=str(checkpoint))
        model.to(device=self.device)
        model.eval()
        self.predictor = SamPredictor(model)

    def predict_box(self, image_rgb, box_xyxy):
        self.predictor.set_image(image_rgb)
        masks, scores, _ = self.predictor.predict(
            box=np.array(box_xyxy, dtype=np.float32),
            multimask_output=False,
        )
        return masks[0], float(scores[0])

# FastSAM is called through the Ultralytics interface.
class FastSAMBoxRunner:
    def __init__(self, checkpoint, device=None):
        from ultralytics import FastSAM
        self.device = device or get_device()
        self.model = FastSAM(str(checkpoint))

    def predict_box(self, image_path, box_xyxy):
        results = self.model(
            str(image_path),
            device=self.device,
            retina_masks=True,
            imgsz=1024,
            conf=0.4,
            iou=0.9,
            bboxes=[box_xyxy],
            verbose=False,
        )
        if results and results[0].masks is not None and len(results[0]) > 0:
            pred_mask = results[0].masks.data[0].detach().cpu().numpy().astype(bool)
        else:
            pred_mask = None
        return pred_mask, float('nan')

# Evaluate one prompt-based model and save one row per selected object.
def run_prompt_model_eval(model_name, records_json, annotations_json, images_dir, checkpoint, output_csv, limit=None):
    from PIL import Image
    records = read_json(records_json)
    if limit is not None:
        records = records[:limit]
    coco = load_coco(annotations_json)
    indices = build_coco_indices(coco)
    device = get_device()

    if model_name == 'SAM':
        runner = SAMBoxRunner(checkpoint, model_type='vit_b', device=device)
    elif model_name == 'SAM_ViT_H':
        runner = SAMBoxRunner(checkpoint, model_type='vit_h', device=device)
    elif model_name == 'MobileSAM':
        runner = MobileSAMBoxRunner(checkpoint, device=device)
    elif model_name == 'FastSAM':
        runner = FastSAMBoxRunner(checkpoint, device=device)
    else:
        raise ValueError(model_name)

    rows = []
    for record in records:
        image_path = Path(images_dir) / record['file_name']
        image_rgb = np.array(Image.open(image_path).convert('RGB'))
        annotation = indices['annotations_by_id'][int(record['annotation_id'])]
        gt_mask = annotation_to_mask(annotation, int(record['height']), int(record['width']))
        # The selected COCO box is the prompt for SAM/MobileSAM/FastSAM.
        box_xyxy = bbox_xywh_to_xyxy(record['bbox_xywh'])

        if model_name == 'FastSAM':
            def predict_once():
                return runner.predict_box(image_path, box_xyxy)
        else:
            def predict_once():
                return runner.predict_box(image_rgb, box_xyxy)

        (pred_mask, score), timing = measure_runtime(predict_once, warmup_runs=1, timed_runs=1)
        rows.append({
            'model': model_name,
            'image_id': int(record['image_id']),
            'file_name': record['file_name'],
            'category_name': record['category_name'],
            'iou': binary_iou(pred_mask, gt_mask),
            'dice': binary_dice(pred_mask, gt_mask),
            'score': score,
            'runtime_mean_seconds': timing['mean_seconds'],
            'runtime_std_seconds': timing['std_seconds'],
            'gt_pixels': int(gt_mask.sum()),
            'pred_pixels': int(np.asarray(pred_mask).astype(bool).sum()) if pred_mask is not None else 0,
        })
    df = pd.DataFrame(rows)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return df


In [ ]:
# This cell evaluates SAM, MobileSAM and FastSAM when the run flag is enabled.
# Otherwise it only checks that the already saved CSV files are available.

if RUN_PROMPT_MODEL_EVAL:
    sam_df = run_prompt_model_eval(
        'SAM',
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        SAM_CHECKPOINT,
        TABLES_DIR / 'sam_val_1000.csv',
    )
    mobilesam_df = run_prompt_model_eval(
        'MobileSAM',
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        MOBILE_SAM_CHECKPOINT,
        TABLES_DIR / 'mobilesam_val_1000.csv',
    )
    fastsam_df = run_prompt_model_eval(
        'FastSAM',
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        FASTSAM_CHECKPOINT,
        TABLES_DIR / 'fastsam_val_1000.csv',
    )
    print_lines('Prompt-model raw evaluation generated', [
        f"SAM rows: {len(sam_df)} -> {TABLES_DIR / 'sam_val_1000.csv'}",
        f"MobileSAM rows: {len(mobilesam_df)} -> {TABLES_DIR / 'mobilesam_val_1000.csv'}",
        f"FastSAM rows: {len(fastsam_df)} -> {TABLES_DIR / 'fastsam_val_1000.csv'}",
    ])
else:
    assert artifact_exists(TABLES_DIR / 'sam_val_1000.csv')
    assert artifact_exists(TABLES_DIR / 'mobilesam_val_1000.csv')
    assert artifact_exists(TABLES_DIR / 'fastsam_val_1000.csv')

In [ ]:
# SAM ViT-H is evaluated separately because it is much heavier and slower to run.

if RUN_SAM_VIT_H_EVAL:
    sam_vit_h_df = run_prompt_model_eval(
        'SAM_ViT_H',
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        SAM_VIT_H_CHECKPOINT,
        TABLES_DIR / 'sam_vit_h_val_1000.csv',
    )
    print(f"SAM ViT-H rows: {len(sam_vit_h_df)} -> {TABLES_DIR / 'sam_vit_h_val_1000.csv'}")
else:
    assert artifact_exists(TABLES_DIR / 'sam_vit_h_val_1000.csv')

## 7. Step 5: DeepLabV3 Pretrained Baseline Evaluation Code


In [ ]:
# DeepLabV3 predicts a full semantic class map instead of a prompted object mask.
# For IoU, I compare only the predicted pixels of the selected target class.

# Load torchvision DeepLabV3-ResNet50 with pretrained weights when requested.
def build_deeplabv3(pretrained=True):
    import torch
    from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights, deeplabv3_resnet50
    weights = DeepLabV3_ResNet50_Weights.DEFAULT if pretrained else None
    model = deeplabv3_resnet50(weights=weights)
    return model

# Load my fine-tuned/model-soup checkpoint into the same architecture.
def load_deeplab_checkpoint_into_model(model, checkpoint_path, device):
    import torch
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state)
    return model

# DeepLabV3 expects ImageNet-style normalization.
def preprocess_for_deeplab(image_rgb):
    import torch
    image = image_rgb.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    image = (image - mean) / std
    return torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).float()

# Evaluate DeepLabV3 by extracting the predicted mask of the target class.
def run_deeplab_object_iou_eval(records_json, annotations_json, images_dir, output_csv, checkpoint=None, model_name='DeepLabV3_pretrained', limit=None):
    import torch
    import torch.nn.functional as F
    from PIL import Image

    records = read_json(records_json)
    if limit is not None:
        records = records[:limit]
    coco = load_coco(annotations_json)
    indices = build_coco_indices(coco)
    device = torch.device(get_device())
    model = build_deeplabv3(pretrained=True)
    if checkpoint is not None:
        model = load_deeplab_checkpoint_into_model(model, checkpoint, device)
    model.to(device)
    model.eval()

    rows = []
    with torch.inference_mode():
        for record in records:
            image_path = Path(images_dir) / record['file_name']
            image_rgb = np.array(Image.open(image_path).convert('RGB'))
            h, w = image_rgb.shape[:2]
            annotation = indices['annotations_by_id'][int(record['annotation_id'])]
            gt_mask = annotation_to_mask(annotation, h, w)
            target_label = COCO_TO_VOC_INDEX.get(record['category_name'])

            start = time.perf_counter()
            tensor = preprocess_for_deeplab(image_rgb).to(device)
            logits = model(tensor)['out']
            logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
            pred_label_map = logits.argmax(dim=1).squeeze(0).detach().cpu().numpy().astype(np.uint8)
            elapsed = time.perf_counter() - start

            if target_label is None:
                iou = np.nan
                pred_pixels = np.nan
            else:
                pred_mask = pred_label_map == int(target_label)
                iou = binary_iou(pred_mask, gt_mask)
                pred_pixels = int(pred_mask.sum())

            rows.append({
                'model': model_name,
                'image_id': int(record['image_id']),
                'category_name': record['category_name'],
                'supported_deeplab_class': target_label is not None,
                'target_label': target_label,
                'iou': iou,
                'runtime_mean_seconds': elapsed,
                'gt_pixels': int(gt_mask.sum()),
                'pred_pixels': pred_pixels,
            })
    df = pd.DataFrame(rows)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return df

def summarize_object_iou(df, model_name):
    return {
        'model': model_name,
        'records_total': len(df),
        'records_with_iou': int(df['iou'].notna().sum()),
        'mean_iou': float(df['iou'].mean()),
        'median_iou': float(df['iou'].median()),
        'std_iou': float(df['iou'].std()),
        'mean_time_seconds': float(df['runtime_mean_seconds'].mean()),
        'median_time_seconds': float(df['runtime_mean_seconds'].median()),
        'total_time_seconds': float(df['runtime_mean_seconds'].sum()),
    }


In [ ]:
# This gives the pretrained DeepLabV3 baseline before any project fine-tuning.

if RUN_DEEPLAB_BASELINE_EVAL:
    deeplab_pretrained_df = run_deeplab_object_iou_eval(
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        TABLES_DIR / 'deeplabv3_pretrained_val_1000.csv',
        checkpoint=None,
        model_name='DeepLabV3_pretrained',
    )
    summary = summarize_object_iou(deeplab_pretrained_df, 'DeepLabV3_pretrained')
    print_lines('DeepLabV3 pretrained raw evaluation generated', [
        f"Rows: {len(deeplab_pretrained_df)} -> {TABLES_DIR / 'deeplabv3_pretrained_val_1000.csv'}",
        f"Mean IoU from this rerun: {summary['mean_iou']:.4f}",
    ])
else:
    assert artifact_exists(TABLES_DIR / 'deeplabv3_pretrained_val_1000.csv')

## 8. Step 6: DeepLabV3 Fine-Tuning Dataset and Loss Code


In [ ]:
# Dataset used for supervised DeepLabV3 fine-tuning.
# Images are resized for training, while masks use nearest-neighbor resizing to keep class IDs valid.

# PyTorch-style dataset that returns image tensor, semantic mask and record metadata.
class CocoDeepLabDataset:
    def __init__(self, records_json, annotations_json, images_dir, image_size=384, augment=False, ignore_index=255):
        import torch
        from torch.utils.data import Dataset
        self.records = read_json(records_json)
        self.coco = load_coco(annotations_json)
        self.indices = build_coco_indices(self.coco)
        self.images_dir = Path(images_dir)
        self.image_size = int(image_size)
        self.augment = bool(augment)
        self.ignore_index = int(ignore_index)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        import cv2
        import torch
        from PIL import Image
        record = self.records[idx]
        image_path = self.images_dir / record['file_name']
        image = np.array(Image.open(image_path).convert('RGB'))
        height, width = image.shape[:2]
        annotations = self.indices['annotations_by_image'].get(int(record['image_id']), [])
        # Build the semantic target mask from all compatible annotations in this image.
        mask = build_semantic_mask_for_image(
            annotations,
            height=height,
            width=width,
            categories=self.indices['categories'],
            ignore_index=self.ignore_index,
        )
        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        # Simple horizontal flip augmentation for training only.
        if self.augment and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
        # Normalize image in the same way as torchvision pretrained models.
        image = image.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        image = (image - mean) / std
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).contiguous().float()
        mask_tensor = torch.from_numpy(mask.astype(np.int64))
        return image_tensor, mask_tensor, record


def deeplab_collate_fn(batch):
    import torch
    images, masks, records = zip(*batch)
    return torch.stack(images, dim=0), torch.stack(masks, dim=0), list(records)

# Control which parts of the pretrained model are allowed to update.
def freeze_for_mode(model, mode):
    for param in model.parameters():
        param.requires_grad = False
    if mode == 'head_only':
        modules = [model.classifier]
        if getattr(model, 'aux_classifier', None) is not None:
            modules.append(model.aux_classifier)
    elif mode == 'layer4_plus_head':
        modules = [model.backbone.layer4, model.classifier]
        if getattr(model, 'aux_classifier', None) is not None:
            modules.append(model.aux_classifier)
    elif mode == 'all':
        modules = [model]
    else:
        raise ValueError(mode)
    for module in modules:
        for param in module.parameters():
            param.requires_grad = True
    return model

# Keep BatchNorm statistics stable when batch size is very small.
def freeze_batch_norm(model):
    import torch
    for module in model.modules():
        if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
            module.eval()
            for param in module.parameters():
                param.requires_grad = False
    return model

# Dice loss focuses on overlap for foreground classes, not just background accuracy.
def foreground_dice_loss(logits, target, ignore_index=255, eps=1e-6):
    import torch
    import torch.nn.functional as F
    valid = target != ignore_index
    if valid.sum() == 0:
        return logits.sum() * 0.0
    num_classes = logits.shape[1]
    probs = F.softmax(logits, dim=1)
    target_clamped = target.clone()
    target_clamped[~valid] = 0
    one_hot = F.one_hot(target_clamped, num_classes=num_classes).permute(0, 3, 1, 2).float()
    valid = valid.unsqueeze(1)
    probs = probs * valid
    one_hot = one_hot * valid
    foreground = torch.arange(num_classes, device=logits.device) > 0
    probs_fg = probs[:, foreground]
    one_hot_fg = one_hot[:, foreground]
    intersection = (probs_fg * one_hot_fg).sum(dim=(0, 2, 3))
    union = probs_fg.sum(dim=(0, 2, 3)) + one_hot_fg.sum(dim=(0, 2, 3))
    dice = (2.0 * intersection + eps) / (union + eps)
    present = one_hot_fg.sum(dim=(0, 2, 3)) > 0
    if present.sum() == 0:
        return logits.sum() * 0.0
    return 1.0 - dice[present].mean()

# Lower background weight because background pixels dominate segmentation masks.
def make_class_weights(background_weight=0.15):
    import torch
    weights = torch.ones(len(VOC_CATEGORIES), dtype=torch.float32)
    weights[0] = float(background_weight)
    return weights


## 10. Training Code Used for the Trials

In [ ]:
def build_optimizer_for_mode(model, mode, lr_head, lr_backbone, weight_decay):
    import torch
    if mode == 'head_only':
        return torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr_head,
            weight_decay=weight_decay,
        )
    if mode == 'layer4_plus_head':
        backbone_params = [p for p in model.backbone.layer4.parameters() if p.requires_grad]
        head_params = list(model.classifier.parameters())
        if getattr(model, 'aux_classifier', None) is not None:
            head_params += list(model.aux_classifier.parameters())
        return torch.optim.AdamW([
            {'params': backbone_params, 'lr': lr_backbone if lr_backbone is not None else lr_head},
            {'params': head_params, 'lr': lr_head},
        ], weight_decay=weight_decay)
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr_head,
        weight_decay=weight_decay,
    )

def build_stage_plan(mode, epochs, staged_plan=None):
    if staged_plan is not None:
        return staged_plan
    if mode != 'staged_head_then_layer4':
        return [{'stage': mode, 'mode': mode, 'epochs': int(epochs)}]
    total_epochs = 2 if epochs == 'staged' else int(epochs)
    head_epochs = max(1, total_epochs // 2)
    layer4_epochs = max(1, total_epochs - head_epochs)
    return [
        {'stage': 'head_warmup', 'mode': 'head_only', 'epochs': head_epochs},
        {'stage': 'layer4_adaptation', 'mode': 'layer4_plus_head', 'epochs': layer4_epochs},
    ]

def compute_deeplab_training_loss(logits, masks, ce_loss, loss_name, dice_weight):
    import torch.nn.functional as F
    if logits.shape[-2:] != masks.shape[-2:]:
        logits = F.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
    loss = ce_loss(logits, masks)
    if loss_name == 'weighted_ce_plus_dice':
        loss = loss + dice_weight * foreground_dice_loss(logits, masks)
    return loss

def evaluate_training_validation_loss(model, val_loader, device, ce_loss, loss_name, dice_weight, frozen_batch_norm=False):
    import torch
    if frozen_batch_norm:
        freeze_batch_norm(model)
    model.eval()
    running_loss = 0.0
    item_count = 0
    with torch.inference_mode():
        for images, masks, _record in val_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            logits = model(images)['out']
            loss = compute_deeplab_training_loss(logits, masks, ce_loss, loss_name, dice_weight)
            running_loss += float(loss.detach().cpu()) * images.shape[0]
            item_count += images.shape[0]
    return running_loss / max(1, item_count)

def train_deeplab_one_experiment(
    records_json,
    annotations_json,
    images_dir,
    output_checkpoint,
    epochs=1,
    image_size=384,
    batch_size=2,
    mode='head_only',
    lr_head=1e-4,
    lr_backbone=None,
    weight_decay=1e-4,
    loss_name='cross_entropy',
    background_weight=0.15,
    dice_weight=0.5,
    gradient_accumulation_steps=1,
    frozen_batch_norm=False,
    workers=0,
    staged_plan=None,
    log_csv=None,
):
    import torch
    from torch.utils.data import DataLoader, random_split

    dataset = CocoDeepLabDataset(records_json, annotations_json, images_dir, image_size=image_size, augment=True)
    val_size = max(1, min(100, int(0.05 * len(dataset))))
    train_size = len(dataset) - val_size
    generator = torch.Generator().manual_seed(SEED)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True, collate_fn=deeplab_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True, collate_fn=deeplab_collate_fn)

    device = torch.device(get_device())
    model = build_deeplabv3(pretrained=True).to(device)
    ce_weights = make_class_weights(background_weight).to(device)
    ce_loss = torch.nn.CrossEntropyLoss(ignore_index=255, weight=ce_weights if loss_name == 'weighted_ce_plus_dice' else None)

    stage_plan = build_stage_plan(mode, epochs, staged_plan=staged_plan)
    log_rows = []
    best_val_loss = float('inf')
    output_checkpoint = Path(output_checkpoint)
    output_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    global_epoch = 0

    for stage in stage_plan:
        active_mode = stage['mode']
        stage_name = stage.get('stage', active_mode)
        model = freeze_for_mode(model, active_mode)
        if frozen_batch_norm:
            model = freeze_batch_norm(model)
        optimizer = build_optimizer_for_mode(model, active_mode, lr_head, lr_backbone, weight_decay)
        stage_epochs = int(stage['epochs'])

        for stage_epoch in range(1, stage_epochs + 1):
            global_epoch += 1
            model.train()
            if frozen_batch_norm:
                freeze_batch_norm(model)
            running_loss = 0.0
            item_count = 0
            optimizer.zero_grad(set_to_none=True)

            for step, (images, masks, _record) in enumerate(train_loader, start=1):
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                logits = model(images)['out']
                loss = compute_deeplab_training_loss(logits, masks, ce_loss, loss_name, dice_weight)
                (loss / gradient_accumulation_steps).backward()
                if step % gradient_accumulation_steps == 0 or step == len(train_loader):
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                running_loss += float(loss.detach().cpu()) * images.shape[0]
                item_count += images.shape[0]

            train_loss = running_loss / max(1, item_count)
            val_loss = evaluate_training_validation_loss(
                model,
                val_loader,
                device,
                ce_loss,
                loss_name,
                dice_weight,
                frozen_batch_norm=frozen_batch_norm,
            )
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'epoch': global_epoch,
                    'stage': stage_name,
                    'stage_epoch': stage_epoch,
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'num_classes': len(VOC_CATEGORIES),
                    'classes': VOC_CATEGORIES,
                    'record_file': str(records_json),
                    'image_size': image_size,
                }, output_checkpoint)
            log_rows.append({
                'epoch': global_epoch,
                'stage': stage_name,
                'stage_epoch': stage_epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'checkpoint': str(output_checkpoint.relative_to(ROOT) if output_checkpoint.is_relative_to(ROOT) else output_checkpoint),
            })
            if log_csv is not None:
                log_csv = Path(log_csv)
                log_csv.parent.mkdir(parents=True, exist_ok=True)
                pd.DataFrame(log_rows).to_csv(log_csv, index=False)
            print(f'epoch {global_epoch} stage={stage_name} train_loss={train_loss:.4f} val_loss={val_loss:.4f}')

    return pd.DataFrame(log_rows)

## 11. Trial A - Start Small: Train Only the Heads

I first kept the pretrained backbone frozen and changed only the segmentation heads.

In [ ]:
trial_name = 'A'
records_json = SELECTED_DIR / 'train_2000_deeplabv3.json'
output_checkpoint = CHECKPOINT_DIR / 'deeplabv3_trial_A_head_only_train2000_epoch1.pth'
loss_log_csv = TABLES_DIR / 'deeplabv3_trial_A_head_only_train2000_epoch1_loss_log.csv'
eval_csv = TABLES_DIR / 'deeplabv3_trial_A_head_only_train2000_epoch1_val1000.csv'

trial_log_df = train_deeplab_one_experiment(
    records_json,
    COCO_TRAIN_ANN,
    COCO_TRAIN_IMAGES,
    output_checkpoint,
    epochs=1,
    image_size=320,
    batch_size=4,
    mode='head_only',
    lr_head=1e-3,
    lr_backbone=None,
    loss_name='cross_entropy',
    log_csv=loss_log_csv,
)

trial_eval_df = run_deeplab_object_iou_eval(
    SELECTED_DIR / 'val_1000_primary_objects.json',
    COCO_VAL_ANN,
    COCO_VAL_IMAGES,
    eval_csv,
    checkpoint=output_checkpoint,
    model_name='DeepLabV3_trial_A',
)

## 12. Trial B - Unfreeze Layer4 on the Same 2,000 Images

After the head-only run, I tried adapting the last ResNet block as well.

In [ ]:
trial_name = 'B'
records_json = SELECTED_DIR / 'train_2000_deeplabv3.json'
output_checkpoint = CHECKPOINT_DIR / 'deeplabv3_trial_B_layer4_balanced_train2000_epoch3.pth'
loss_log_csv = TABLES_DIR / 'deeplabv3_trial_B_layer4_balanced_train2000_epoch3_loss_log.csv'
eval_csv = TABLES_DIR / 'deeplabv3_trial_B_layer4_balanced_train2000_epoch3_val1000.csv'

trial_log_df = train_deeplab_one_experiment(
    records_json,
    COCO_TRAIN_ANN,
    COCO_TRAIN_IMAGES,
    output_checkpoint,
    epochs=3,
    image_size=384,
    batch_size=2,
    mode='layer4_plus_head',
    lr_head=2e-4,
    lr_backbone=5e-5,
    loss_name='weighted_ce_plus_dice',
    log_csv=loss_log_csv,
)

trial_eval_df = run_deeplab_object_iou_eval(
    SELECTED_DIR / 'val_1000_primary_objects.json',
    COCO_VAL_ANN,
    COCO_VAL_IMAGES,
    eval_csv,
    checkpoint=output_checkpoint,
    model_name='DeepLabV3_trial_B',
)

## 13. Trial C - Use More Training Images with Staged Fine-Tuning

Then I increased the training subset and used the staged head-then-layer4 schedule.

In [ ]:
trial_name = 'C'
records_json = SELECTED_DIR / 'train_5000_deeplabv3.json'
output_checkpoint = CHECKPOINT_DIR / 'deeplabv3_trial_C_primary_object_resized_train5000.pth'
loss_log_csv = TABLES_DIR / 'deeplabv3_trial_C_primary_object_resized_train5000_loss_log.csv'
eval_csv = TABLES_DIR / 'deeplabv3_trial_C_primary_object_resized_train5000_val1000.csv'

trial_log_df = train_deeplab_one_experiment(
    records_json,
    COCO_TRAIN_ANN,
    COCO_TRAIN_IMAGES,
    output_checkpoint,
    epochs=3,
    image_size=384,
    batch_size=2,
    mode='staged_head_then_layer4',
    lr_head=2e-4,
    lr_backbone=5e-5,
    loss_name='weighted_ce_plus_dice',
    log_csv=loss_log_csv,
)

trial_eval_df = run_deeplab_object_iou_eval(
    SELECTED_DIR / 'val_1000_primary_objects.json',
    COCO_VAL_ANN,
    COCO_VAL_IMAGES,
    eval_csv,
    checkpoint=output_checkpoint,
    model_name='DeepLabV3_trial_C',
)

## 14. Trial D - Larger Subset with Lower Learning Rates

This run kept the staged schedule but used more data and smaller learning rates.

In [ ]:
trial_name = 'D'
records_json = SELECTED_DIR / 'train_10000_deeplabv3.json'
output_checkpoint = CHECKPOINT_DIR / 'deeplabv3_trial_D_staged_train10000_gentle.pth'
loss_log_csv = TABLES_DIR / 'deeplabv3_trial_D_staged_train10000_gentle_loss_log.csv'
eval_csv = TABLES_DIR / 'deeplabv3_trial_D_staged_train10000_gentle_val1000.csv'

trial_log_df = train_deeplab_one_experiment(
    records_json,
    COCO_TRAIN_ANN,
    COCO_TRAIN_IMAGES,
    output_checkpoint,
    epochs='staged',
    image_size=384,
    batch_size=2,
    mode='staged_head_then_layer4',
    lr_head=1e-4,
    lr_backbone=2e-5,
    loss_name='weighted_ce_plus_dice',
    log_csv=loss_log_csv,
)

trial_eval_df = run_deeplab_object_iou_eval(
    SELECTED_DIR / 'val_1000_primary_objects.json',
    COCO_VAL_ANN,
    COCO_VAL_IMAGES,
    eval_csv,
    checkpoint=output_checkpoint,
    model_name='DeepLabV3_trial_D',
)

## 15. Model-Soup Checkpoint Code

In [ ]:
# Model-soup interpolation mixes pretrained and fine-tuned weights.
# This keeps the final checkpoint close to the strong pretrained DeepLabV3 baseline.

def extract_state_dict(checkpoint):
    state = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    return state

def interpolate_state_dict(pretrained_state, finetuned_state, alpha):
    mixed = {}
    for key, pre_value in pretrained_state.items():
        fine_value = finetuned_state.get(key)
        if fine_value is None:
            mixed[key] = pre_value
            continue
        # Interpolate only floating-point tensors; keep integer buffers from the fine-tuned checkpoint.
        if hasattr(pre_value, 'dtype') and str(pre_value.dtype).startswith('torch.float'):
            mixed[key] = (1.0 - alpha) * pre_value + alpha * fine_value
        else:
            mixed[key] = fine_value
    return mixed

def create_deeplab_model_soup_checkpoint(finetuned_checkpoint_path, output_checkpoint_path, alpha):
    import torch
    device = torch.device('cpu')
    pretrained_model = build_deeplabv3(pretrained=True)
    pretrained_state = pretrained_model.state_dict()
    fine_checkpoint = torch.load(finetuned_checkpoint_path, map_location=device)
    fine_state = extract_state_dict(fine_checkpoint)
    mixed_state = interpolate_state_dict(pretrained_state, fine_state, alpha=float(alpha))
    payload = {
        'model_state_dict': mixed_state,
        'alpha': float(alpha),
        'source_finetuned_checkpoint': str(finetuned_checkpoint_path),
        'description': 'Model-soup interpolation between pretrained DeepLabV3 and fine-tuned DeepLabV3.',
    }
    output_checkpoint_path = Path(output_checkpoint_path)
    output_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, output_checkpoint_path)
    return output_checkpoint_path

# Try different alpha values and evaluate each mixed checkpoint.
def run_model_soup_alpha_search(finetuned_checkpoint_path, alpha_values, records_json, annotations_json, images_dir, output_dir):
    search_rows = []
    for alpha in alpha_values:
        soup_path = Path(output_dir) / f'deeplabv3_model_soup_alpha_{alpha:.3f}.pth'
        create_deeplab_model_soup_checkpoint(finetuned_checkpoint_path, soup_path, alpha)
        eval_csv = Path(output_dir) / f'deeplabv3_model_soup_alpha_{alpha:.3f}_val1000.csv'
        df = run_deeplab_object_iou_eval(records_json, annotations_json, images_dir, eval_csv, checkpoint=soup_path, model_name=f'DeepLabV3_model_soup_alpha_{alpha:.3f}')
        search_rows.append(summarize_object_iou(df, f'DeepLabV3_model_soup_alpha_{alpha:.3f}'))
    search_df = pd.DataFrame(search_rows).sort_values('mean_iou', ascending=False)
    return search_df

if RUN_MODEL_SOUP_SEARCH:
    if not artifact_exists(FINAL_RAW_DEEPLAB_CHECKPOINT):
        raise FileNotFoundError(
            f'Missing raw Trial E checkpoint: {FINAL_RAW_DEEPLAB_CHECKPOINT}. '
            'Restore it or rerun Trial E before rerunning model-soup search.'
        )
    search_df = run_model_soup_alpha_search(
        FINAL_RAW_DEEPLAB_CHECKPOINT,
        alpha_values=[0.0, 0.1, 0.18, 0.25, 0.5, 0.75, 1.0],
        records_json=SELECTED_DIR / 'val_1000_primary_objects.json',
        annotations_json=COCO_VAL_ANN,
        images_dir=COCO_VAL_IMAGES,
        output_dir=CHECKPOINT_DIR,
    )
    best = search_df.iloc[0]
    create_deeplab_model_soup_checkpoint(
        FINAL_RAW_DEEPLAB_CHECKPOINT,
        FINAL_DEEPLAB_CHECKPOINT,
        alpha=0.5,
    )
    print_lines('Model-soup alpha search rerun', [
        f"Best model in this rerun: {best['model']}",
        f"Mean IoU: {best['mean_iou']:.4f}",
        f"Saved selected alpha 0.5 checkpoint: {FINAL_DEEPLAB_CHECKPOINT}",
    ])
else:
    assert artifact_exists(FINAL_DEEPLAB_CHECKPOINT)

final_ft_df = read_csv('deeplabv3_final_finetuning_result.csv')
pretrained_final = final_ft_df.iloc[0]
selected = final_ft_df.iloc[1]
assert np.isclose(selected.delta_vs_pretrained, selected.mean_iou - pretrained_final.mean_iou)

## 16. Trial E - Final 50,000 Image Checkpoint

After the smaller trials, I used the larger 50,000-image run as the final DeepLabV3 checkpoint.

In [ ]:
trial_name = 'E'
records_json = SELECTED_DIR / 'train_50000_deeplabv3.json'
raw_trial_E_checkpoint = CHECKPOINT_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2.pth'
trial_E_log_csv = TABLES_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_loss_log.csv'
final_trial_E_checkpoint = FINAL_DEEPLAB_CHECKPOINT
final_eval_csv = TABLES_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv'

trial_E_log_df = train_deeplab_one_experiment(
    records_json,
    COCO_TRAIN_ANN,
    COCO_TRAIN_IMAGES,
    raw_trial_E_checkpoint,
    epochs='staged',
    image_size=384,
    batch_size=1,
    mode='staged_head_then_layer4',
    lr_head=1e-4,
    lr_backbone=2e-5,
    loss_name='weighted_ce_plus_dice',
    gradient_accumulation_steps=2,
    frozen_batch_norm=True,
    log_csv=trial_E_log_csv,
)

create_deeplab_model_soup_checkpoint(
    raw_trial_E_checkpoint,
    final_trial_E_checkpoint,
    alpha=0.5,
)

deeplab_finetuned_df = run_deeplab_object_iou_eval(
    SELECTED_DIR / 'val_1000_primary_objects.json',
    COCO_VAL_ANN,
    COCO_VAL_IMAGES,
    final_eval_csv,
    checkpoint=final_trial_E_checkpoint,
    model_name='DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5',
)

## 17. Final DeepLabV3 Evaluation

In [ ]:
# Evaluate the final selected fine-tuned checkpoint on the same 1000 records as the baseline.

if RUN_DEEPLAB_FINETUNED_EVAL:
    deeplab_finetuned_df = run_deeplab_object_iou_eval(
        SELECTED_DIR / 'val_1000_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        TABLES_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv',
        checkpoint=FINAL_DEEPLAB_CHECKPOINT,
        model_name='DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5',
    )
    summary = summarize_object_iou(deeplab_finetuned_df, 'DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5')
    print_lines('Final DeepLabV3 raw evaluation generated', [
        f"Rows: {len(deeplab_finetuned_df)} -> {TABLES_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv'}",
        f"Mean IoU from this rerun: {summary['mean_iou']:.4f}",
    ])
else:
    assert artifact_exists(TABLES_DIR / 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv')

## 18. Summary Tables from Raw Results

In [ ]:
# Rebuild summary tables from raw per-image CSV files instead of typing result values manually.

def summarize_raw_result_csv(csv_name, model_name, total_records):
    df = read_csv(csv_name)
    # NaN IoU values are ignored here, for example unsupported DeepLabV3 classes.
    iou = pd.to_numeric(df['iou'], errors='coerce')
    time_col = pd.to_numeric(df['runtime_mean_seconds'], errors='coerce')
    return {
        'model': model_name,
        'records_total': int(total_records),
        'records_with_iou': int(iou.notna().sum()),
        'mean_iou': float(iou.mean()),
        'median_iou': float(iou.median()),
        'std_iou': float(iou.std()),
        'mean_time_seconds': float(time_col.mean()),
        'median_time_seconds': float(time_col.median()),
        'total_time_seconds': float(time_col.sum()),
        'source_csv': f'results/tables/{csv_name}',
    }

def rebuild_summary_tables_from_raw_csvs():
    val1000_total = len(read_json(SELECTED_DIR / 'val_1000_primary_objects.json'))
    valall_total = len(read_json(SELECTED_DIR / 'val_all_primary_objects.json'))

    # Main comparison: all models on the same 1000 selected validation records.
    rows_1000 = [
        summarize_raw_result_csv('sam_val_1000.csv', 'SAM_ViT_B', val1000_total),
        summarize_raw_result_csv('sam_vit_h_val_1000.csv', 'SAM_ViT_H', val1000_total),
        summarize_raw_result_csv('mobilesam_val_1000.csv', 'MobileSAM', val1000_total),
        summarize_raw_result_csv('fastsam_val_1000.csv', 'FastSAM', val1000_total),
        summarize_raw_result_csv('deeplabv3_pretrained_val_1000.csv', 'DeepLabV3_pretrained', val1000_total),
        summarize_raw_result_csv(
            'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv',
            'DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5',
            val1000_total,
        ),
    ]
    summary_1000 = pd.DataFrame(rows_1000)
    summary_1000.to_csv(TABLES_DIR / 'final_report_all_model_val_1000_summary.csv', index=False)
    summary_1000[
        ['model', 'records_total', 'records_with_iou', 'mean_time_seconds', 'median_time_seconds', 'total_time_seconds']
    ].to_csv(TABLES_DIR / 'final_speed_table.csv', index=False)

    # Larger check: efficient models and DeepLabV3 before/after on val_all.
    rows_all = [
        summarize_raw_result_csv('mobilesam_val_all.csv', 'MobileSAM', valall_total),
        summarize_raw_result_csv('fastsam_val_all.csv', 'FastSAM', valall_total),
        summarize_raw_result_csv('deeplabv3_pretrained_val_all.csv', 'DeepLabV3_pretrained', valall_total),
        summarize_raw_result_csv(
            'deeplabv3_finetuned_best_val_all.csv',
            'DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5',
            valall_total,
        ),
    ]
    summary_all = pd.DataFrame(rows_all).rename(columns={'records_with_iou': 'records_evaluated_for_iou'})
    summary_all['throughput_images_per_second'] = summary_all['records_total'] / summary_all['total_time_seconds']
    summary_all.to_csv(TABLES_DIR / 'final_report_all_model_val_all_summary.csv', index=False)
    summary_all[
        ['model', 'records_total', 'mean_time_seconds', 'median_time_seconds', 'total_time_seconds',
         'throughput_images_per_second', 'source_csv']
    ].to_csv(TABLES_DIR / 'final_speed_table_val_all.csv', index=False)

    # DeepLabV3 before/after table for the larger validation check.
    deeplab_all = summary_all[summary_all['model'].str.contains('DeepLabV3', regex=False)].copy()
    pre_all = deeplab_all[deeplab_all['model'].eq('DeepLabV3_pretrained')].iloc[0]
    deeplab_all.insert(0, 'validation_split', 'val_all_4844')
    deeplab_all['delta_mean_iou_vs_pretrained'] = deeplab_all['mean_iou'] - float(pre_all.mean_iou)
    deeplab_all['delta_median_iou_vs_pretrained'] = deeplab_all['median_iou'] - float(pre_all.median_iou)
    deeplab_all.to_csv(TABLES_DIR / 'deeplabv3_val_all_before_after_summary.csv', index=False)

    pre_1000 = summary_1000[summary_1000['model'].eq('DeepLabV3_pretrained')].iloc[0]
    ft_1000 = summary_1000[summary_1000['model'].str.contains('finetuned', regex=False)].iloc[0]
    pd.DataFrame([
        {
            'model': 'DeepLabV3_pretrained_fullres',
            'mean_iou': float(pre_1000.mean_iou),
            'delta_vs_pretrained': 0.0,
            'source_csv': 'results/tables/deeplabv3_pretrained_val_1000.csv',
        },
        {
            'model': 'DeepLabV3_model_soup_alpha_0.5_train50000_frozenbn_accum2',
            'mean_iou': float(ft_1000.mean_iou),
            'delta_vs_pretrained': float(ft_1000.mean_iou - pre_1000.mean_iou),
            'source_csv': 'results/tables/deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv',
        },
    ]).to_csv(TABLES_DIR / 'deeplabv3_final_finetuning_result.csv', index=False)

    return summary_1000, summary_all, deeplab_all

if REBUILD_SUMMARY_TABLES:
    summary_1000, summary_all, deeplab_all = rebuild_summary_tables_from_raw_csvs()
    generated_summary_files = [
        TABLES_DIR / 'final_report_all_model_val_1000_summary.csv',
        TABLES_DIR / 'final_speed_table.csv',
        TABLES_DIR / 'final_report_all_model_val_all_summary.csv',
        TABLES_DIR / 'final_speed_table_val_all.csv',
        TABLES_DIR / 'deeplabv3_val_all_before_after_summary.csv',
        TABLES_DIR / 'deeplabv3_final_finetuning_result.csv',
    ]
    assert all(artifact_exists(path) for path in generated_summary_files)
    print('Generated summary files:', [str(path.relative_to(ROOT)) for path in generated_summary_files])
else:
    assert artifact_exists(TABLES_DIR / 'final_report_all_model_val_1000_summary.csv')
    assert artifact_exists(TABLES_DIR / 'final_report_all_model_val_all_summary.csv')

## 19. Result Figures

In [ ]:
# Generate report figures directly from the saved result tables.
# This keeps the figures consistent with the CSV values used in the report.

summary_1000 = read_csv('final_report_all_model_val_1000_summary.csv')

figure_model_names = {
    'SAM_ViT_B': 'SAM ViT-B',
    'SAM_ViT_H': 'SAM ViT-H',
    'MobileSAM': 'MobileSAM',
    'FastSAM': 'FastSAM',
    'DeepLabV3_pretrained': 'DeepLabV3\npretrained',
    'DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5': 'DeepLabV3\nfine-tuned',
}
figure_colors = {
    'SAM ViT-B': '#4E79A7',
    'SAM ViT-H': '#376795',
    'MobileSAM': '#59A14F',
    'FastSAM': '#F28E2B',
    'DeepLabV3\npretrained': '#8CD17D',
    'DeepLabV3\nfine-tuned': '#2E7D32',
}
figure_order = list(figure_model_names.keys())
plot_df = summary_1000.set_index('model').loc[figure_order].reset_index()
plot_df['label'] = plot_df['model'].map(figure_model_names)
plot_df['color'] = plot_df['label'].map(figure_colors)

# Small plotting helper for cleaner report figures.
def clean_axis(ax, grid_axis):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis=grid_axis, color='#D9D9D9', linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)

def save_horizontal_bar_figure(value_column, xlabel, title, output_name, value_format):
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    fig.subplots_adjust(left=0.25, right=0.94, top=0.84, bottom=0.16)
    y_positions = range(len(plot_df))
    values = plot_df[value_column].astype(float)
    ax.barh(y_positions, values, color=plot_df['color'], height=0.62)
    ax.set_yticks(y_positions, plot_df['label'])
    ax.invert_yaxis()
    ax.set_xlim(0, values.max() * 1.18 if value_column != 'mean_iou' else 0.95)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    clean_axis(ax, 'x')
    offset = 0.012 if value_column == 'mean_iou' else 0.15
    for y, value in zip(y_positions, values):
        ax.text(value + offset, y, value_format.format(value), va='center', ha='left', fontsize=10.5, color='#222222')
    output_path = FIGURES_DIR / output_name
    fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return output_path

# Check that the DeepLabV3 summary table matches the raw evaluation CSVs.
def deeplab_summary_values():
    pre = summary_1000[summary_1000['model'].eq('DeepLabV3_pretrained')].iloc[0]
    fine = summary_1000[summary_1000['model'].str.contains('finetuned', regex=False)].iloc[0]
    final_ft = read_csv('deeplabv3_final_finetuning_result.csv')
    assert np.isclose(float(pre['mean_iou']), float(final_ft.iloc[0]['mean_iou']))
    assert np.isclose(float(fine['mean_iou']), float(final_ft.iloc[1]['mean_iou']))
    return float(pre['mean_iou']), float(fine['mean_iou'])

# Pair pretrained and fine-tuned DeepLabV3 rows to see image-level changes.
def build_deeplab_paired_iou_tables():
    pre = read_csv('deeplabv3_pretrained_val_1000.csv')
    fine = read_csv('deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv')
    key_cols = ['image_id', 'file_name', 'category_name']
    pre = pre[pre['supported_deeplab_class'].astype(bool)].copy()
    fine = fine[fine['supported_deeplab_class'].astype(bool)].copy()
    paired = pre[key_cols + ['iou', 'runtime_mean_seconds', 'gt_pixels', 'pred_pixels']].merge(
        fine[key_cols + ['iou', 'runtime_mean_seconds', 'pred_pixels']],
        on=key_cols,
        suffixes=('_pretrained', '_finetuned'),
        how='inner',
    )
    paired['delta_iou'] = paired['iou_finetuned'] - paired['iou_pretrained']
    paired['abs_delta_iou'] = paired['delta_iou'].abs()
    paired = paired.sort_values(['delta_iou', 'image_id']).reset_index(drop=True)
    paired_path = TABLES_DIR / 'deeplabv3_val1000_paired_iou_delta.csv'
    paired.to_csv(paired_path, index=False)

    # Class-wise view shows where fine-tuning helped or hurt.
    classwise = paired.groupby('category_name', as_index=False).agg(
        records=('image_id', 'count'),
        mean_iou_pretrained=('iou_pretrained', 'mean'),
        mean_iou_finetuned=('iou_finetuned', 'mean'),
        median_iou_pretrained=('iou_pretrained', 'median'),
        median_iou_finetuned=('iou_finetuned', 'median'),
        mean_delta_iou=('delta_iou', 'mean'),
    )
    classwise = classwise.sort_values('mean_delta_iou', ascending=False).reset_index(drop=True)
    classwise_path = TABLES_DIR / 'deeplabv3_val1000_classwise_before_after.csv'
    classwise.to_csv(classwise_path, index=False)

    pre_summary, fine_summary = deeplab_summary_values()
    assert len(paired) == int(summary_1000.loc[summary_1000['model'].eq('DeepLabV3_pretrained'), 'records_with_iou'].iloc[0])
    assert np.isclose(paired['iou_pretrained'].mean(), pre_summary)
    assert np.isclose(paired['iou_finetuned'].mean(), fine_summary)
    return paired, classwise, paired_path, classwise_path

def save_deeplab_classwise_figure(classwise, min_records=1):
    classwise_plot = classwise[classwise['records'] >= min_records].sort_values('mean_delta_iou').copy()
    classwise_plot['plot_label'] = classwise_plot.apply(lambda row: f"{row['category_name']} (n={int(row['records'])})", axis=1)
    fig, ax = plt.subplots(figsize=(7.8, 6.4))
    fig.subplots_adjust(left=0.32, right=0.94, top=0.88, bottom=0.12)
    colors = ['#C44E52' if value < 0 else '#2E7D32' for value in classwise_plot['mean_delta_iou']]
    y_positions = range(len(classwise_plot))
    ax.barh(y_positions, classwise_plot['mean_delta_iou'], color=colors, height=0.62)
    ax.axvline(0, color='#333333', linewidth=1.0)
    ax.set_yticks(y_positions, classwise_plot['plot_label'])
    ax.set_xlabel('Mean IoU change')
    ax.set_title('DeepLabV3 Mean IoU Change by Class')
    clean_axis(ax, 'x')
    output_path = FIGURES_DIR / 'report_deeplab_classwise_before_after.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return output_path

paired_iou, classwise_iou, paired_iou_path, classwise_iou_path = build_deeplab_paired_iou_tables()

figure_outputs = [
    save_horizontal_bar_figure(
        'mean_iou',
        'Mean IoU',
        'Segmentation Quality on 1000 COCO Validation Records',
        'report_mean_iou_comparison.png',
        '{:.4f}',
    ),
    save_horizontal_bar_figure(
        'mean_time_seconds',
        'Mean inference time per image (seconds)',
        'Inference Time on 1000 COCO Validation Records',
        'report_runtime_comparison.png',
        '{:.2f}s',
    ),
    save_deeplab_classwise_figure(classwise_iou),
]

for output_path in [paired_iou_path, classwise_iou_path, *figure_outputs]:
    assert artifact_exists(output_path)
    print(f'Generated {output_path.relative_to(ROOT)}')

## 20. Visual Segmentation Example

In [ ]:
# Choose a visual example from measured results, then redraw each model mask for that image.
# The example is selected from the CSVs so it stays connected to the experiment.

def resize_bool_mask(mask, shape_hw):
    from PIL import Image
    if mask is None:
        return np.zeros(shape_hw, dtype=bool)
    mask = np.asarray(mask).astype(bool)
    if mask.shape == tuple(shape_hw):
        return mask
    resized = Image.fromarray(mask.astype(np.uint8) * 255).resize((shape_hw[1], shape_hw[0]), resample=Image.Resampling.NEAREST)
    return np.asarray(resized) > 0

# Recreate a DeepLabV3 mask for one selected visual example.
def deeplab_mask_for_record(record, checkpoint=None, device_name='cpu'):
    import torch
    import torch.nn.functional as F
    from PIL import Image
    image_rgb = np.array(Image.open(COCO_VAL_IMAGES / record['file_name']).convert('RGB'))
    h, w = image_rgb.shape[:2]
    target_label = COCO_TO_VOC_INDEX.get(record['category_name'])
    if target_label is None:
        return np.zeros((h, w), dtype=bool)
    device = torch.device(device_name)
    model = build_deeplabv3(pretrained=True)
    if checkpoint is not None:
        model = load_deeplab_checkpoint_into_model(model, checkpoint, device)
    model.to(device)
    model.eval()
    with torch.inference_mode():
        logits = model(preprocess_for_deeplab(image_rgb).to(device))['out']
        logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
        label_map = logits.argmax(dim=1).squeeze(0).detach().cpu().numpy().astype(np.uint8)
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return label_map == int(target_label)

# Generate one qualitative comparison image using the same selected object.
def generate_visual_segmentation_comparison(record, device_name='cpu'):
    import gc
    from PIL import Image
    image_path = COCO_VAL_IMAGES / record['file_name']
    image_rgb = np.array(Image.open(image_path).convert('RGB'))
    h, w = image_rgb.shape[:2]
    coco = load_coco(COCO_VAL_ANN)
    indices = build_coco_indices(coco)
    annotation = indices['annotations_by_id'][int(record['annotation_id'])]
    gt_mask = annotation_to_mask(annotation, h, w)
    box_xyxy = bbox_xywh_to_xyxy(record['bbox_xywh'])

    # Store all masks in the same list so they can be plotted side by side.
    predictions = [('Ground truth', gt_mask)]

    sam_runner = SAMBoxRunner(SAM_CHECKPOINT, model_type='vit_b', device=device_name)
    sam_mask, _ = sam_runner.predict_box(image_rgb, box_xyxy)
    predictions.append(('SAM ViT-B', resize_bool_mask(sam_mask, (h, w))))
    del sam_runner
    gc.collect()

    sam_h_runner = SAMBoxRunner(SAM_VIT_H_CHECKPOINT, model_type='vit_h', device=device_name)
    sam_h_mask, _ = sam_h_runner.predict_box(image_rgb, box_xyxy)
    predictions.append(('SAM ViT-H', resize_bool_mask(sam_h_mask, (h, w))))
    del sam_h_runner
    gc.collect()

    mobile_runner = MobileSAMBoxRunner(MOBILE_SAM_CHECKPOINT, device=device_name)
    mobile_mask, _ = mobile_runner.predict_box(image_rgb, box_xyxy)
    predictions.append(('MobileSAM', resize_bool_mask(mobile_mask, (h, w))))
    del mobile_runner
    gc.collect()

    fast_runner = FastSAMBoxRunner(FASTSAM_CHECKPOINT, device=device_name)
    fast_mask, _ = fast_runner.predict_box(image_path, box_xyxy)
    predictions.append(('FastSAM', resize_bool_mask(fast_mask, (h, w))))
    del fast_runner
    gc.collect()

    predictions.append(('DeepLabV3 pretrained', deeplab_mask_for_record(record, checkpoint=None, device_name=device_name)))
    predictions.append(('DeepLabV3 fine-tuned', deeplab_mask_for_record(record, checkpoint=FINAL_DEEPLAB_CHECKPOINT, device_name=device_name)))

    rows = []
    # Save the IoU values used to label/check this visual example.
    for model_name, pred_mask in predictions[1:]:
        rows.append({
            'model': model_name,
            'image_id': int(record['image_id']),
            'category_name': record['category_name'],
            'example_iou': binary_iou(pred_mask, gt_mask),
            'pred_pixels': int(pred_mask.sum()),
            'gt_pixels': int(gt_mask.sum()),
        })

    ncols = 4
    nrows = int(np.ceil(len(predictions) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.asarray(axes).ravel()
    for ax, (model_name, mask) in zip(axes, predictions):
        ax.imshow(image_rgb)
        ax.imshow(np.ma.masked_where(~mask, mask), cmap='autumn', alpha=0.45)
        if model_name == 'Ground truth':
            title = f"{model_name}\n{record['category_name']}"
        else:
            iou_value = next(row['example_iou'] for row in rows if row['model'] == model_name)
            title = f'{model_name}\nIoU={iou_value:.3f}'
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[len(predictions):]:
        ax.axis('off')
    plt.tight_layout()

    figure_output = FIGURES_DIR / 'final_visual_segmentation_comparison.png'
    table_output = TABLES_DIR / 'final_visual_example_iou.csv'
    fig.savefig(figure_output, dpi=160, bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(table_output, index=False)
    return figure_output, table_output

def select_visual_comparison_record():
    records = read_json(SELECTED_DIR / 'val_1000_primary_objects.json')
    records_by_key = {(int(r['image_id']), r['category_name']): r for r in records}
    preferred_key = (149770, 'person')
    preferred_record = records_by_key.get(preferred_key)
    raw_files = {
        'SAM ViT-B': 'sam_val_1000.csv',
        'SAM ViT-H': 'sam_vit_h_val_1000.csv',
        'MobileSAM': 'mobilesam_val_1000.csv',
        'FastSAM': 'fastsam_val_1000.csv',
        'DeepLabV3 pretrained': 'deeplabv3_pretrained_val_1000.csv',
        'DeepLabV3 fine-tuned': 'deeplabv3_staged_train50000_gentle_frozenbn_accum2_model_soup_best_val1000.csv',
    }
    merged = None
    for model_name, csv_name in raw_files.items():
        df = read_csv(csv_name)[['image_id', 'category_name', 'iou']].rename(columns={'iou': model_name})
        merged = df if merged is None else merged.merge(df, on=['image_id', 'category_name'], how='inner')

    summary = read_csv('final_report_all_model_val_1000_summary.csv')
    target_iou = {
        'SAM ViT-B': float(summary.loc[summary['model'].eq('SAM_ViT_B'), 'mean_iou'].iloc[0]),
        'SAM ViT-H': float(summary.loc[summary['model'].eq('SAM_ViT_H'), 'mean_iou'].iloc[0]),
        'MobileSAM': float(summary.loc[summary['model'].eq('MobileSAM'), 'mean_iou'].iloc[0]),
        'FastSAM': float(summary.loc[summary['model'].eq('FastSAM'), 'mean_iou'].iloc[0]),
        'DeepLabV3 pretrained': float(summary.loc[summary['model'].eq('DeepLabV3_pretrained'), 'mean_iou'].iloc[0]),
        'DeepLabV3 fine-tuned': float(summary.loc[summary['model'].str.contains('finetuned', regex=False), 'mean_iou'].iloc[0]),
    }

    records_df = pd.DataFrame(records)[['image_id', 'category_name', 'width', 'height', 'area']]
    merged = merged.dropna(subset=list(target_iou)).merge(records_df, on=['image_id', 'category_name'], how='left')
    merged = merged.copy()
    for model_name, target in target_iou.items():
        merged[f'error_{model_name}'] = (merged[model_name] - target).abs()
    error_columns = [f'error_{model_name}' for model_name in target_iou]
    merged['mean_abs_error_to_final_means'] = merged[error_columns].mean(axis=1)
    merged['max_abs_error_to_final_means'] = merged[error_columns].max(axis=1)
    merged['deeplab_delta_iou'] = merged['DeepLabV3 fine-tuned'] - merged['DeepLabV3 pretrained']
    if preferred_record is not None:
        preferred_row = merged[(merged['image_id'].eq(preferred_key[0])) & (merged['category_name'].eq(preferred_key[1]))]
        if not preferred_row.empty:
            row = preferred_row.iloc[0]
            return preferred_record, float(row['mean_abs_error_to_final_means']), float(row['deeplab_delta_iou'])

    visually_clear_categories = ['dog', 'cat', 'bird', 'horse', 'sheep', 'cow', 'train', 'bus', 'car', 'bicycle', 'motorcycle']
    merged['visual_category_priority'] = np.where(merged['category_name'].isin(visually_clear_categories), 0, 1)

    # Prefer a clear qualitative example while keeping it tied to the measured results.
    merged = merged[
        (merged['deeplab_delta_iou'] >= 0)
        & (merged['area'] >= 5000)
        & (merged['width'] >= 300)
        & (merged['height'] >= 250)
    ].sort_values(
        ['visual_category_priority', 'mean_abs_error_to_final_means', 'max_abs_error_to_final_means'],
        ascending=True,
    )
    for row in merged.itertuples(index=False):
        key = (int(row.image_id), row.category_name)
        if key in records_by_key:
            return records_by_key[key], float(row.mean_abs_error_to_final_means), float(row.deeplab_delta_iou)
    return records[0], float('nan'), float('nan')

def visual_table_matches_record(path, record):
    if not artifact_exists(path):
        return False
    table = pd.read_csv(path)
    if table.empty:
        return False
    return (
        int(table.iloc[0]['image_id']) == int(record['image_id'])
        and str(table.iloc[0]['category_name']) == str(record['category_name'])
    )

visual_record, visual_mean_error, visual_delta = select_visual_comparison_record()
visual_figure = FIGURES_DIR / 'final_visual_segmentation_comparison.png'
visual_table = TABLES_DIR / 'final_visual_example_iou.csv'
visual_needs_update = (
    REGENERATE_VISUAL_COMPARISON
    or not artifact_exists(visual_figure)
    or not visual_table_matches_record(visual_table, visual_record)
)
if visual_needs_update:
    visual_figure, visual_table = generate_visual_segmentation_comparison(visual_record, device_name='cpu')

assert artifact_exists(visual_figure)
assert artifact_exists(visual_table)
print(
    f"Selected visual example: image_id={visual_record['image_id']}, "
    f"class={visual_record['category_name']}, mean error to final IoUs={visual_mean_error:.3f}, "
    f"DeepLabV3 delta IoU={visual_delta:.3f}"
)
print(f'Visual comparison figure available: {visual_figure.relative_to(ROOT)}')
print(f'Visual example IoU table available: {visual_table.relative_to(ROOT)}')

## 21. Larger Validation Check

In [ ]:
# Larger validation check: repeat important evaluations on more selected val2017 records.
# This checks whether the main conclusion is stable beyond the 1000-record subset.

# This cell is expensive, so it is controlled by a run flag.
if RUN_FULL_VALIDATION_EVAL:
    mobilesam_all = run_prompt_model_eval(
        'MobileSAM',
        SELECTED_DIR / 'val_all_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        MOBILE_SAM_CHECKPOINT,
        TABLES_DIR / 'mobilesam_val_all.csv',
    )
    fastsam_all = run_prompt_model_eval(
        'FastSAM',
        SELECTED_DIR / 'val_all_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        FASTSAM_CHECKPOINT,
        TABLES_DIR / 'fastsam_val_all.csv',
    )
    deeplab_pre_all = run_deeplab_object_iou_eval(
        SELECTED_DIR / 'val_all_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        TABLES_DIR / 'deeplabv3_pretrained_val_all.csv',
        checkpoint=None,
        model_name='DeepLabV3_pretrained',
    )
    deeplab_ft_all = run_deeplab_object_iou_eval(
        SELECTED_DIR / 'val_all_primary_objects.json',
        COCO_VAL_ANN,
        COCO_VAL_IMAGES,
        TABLES_DIR / 'deeplabv3_finetuned_best_val_all.csv',
        checkpoint=FINAL_DEEPLAB_CHECKPOINT,
        model_name='DeepLabV3_finetuned_best_model_soup_train50000_alpha_0.5',
    )
    print('Generated full-validation raw CSV files:')
    print(f"- {TABLES_DIR / 'mobilesam_val_all.csv'} rows={len(mobilesam_all)}")
    print(f"- {TABLES_DIR / 'fastsam_val_all.csv'} rows={len(fastsam_all)}")
    print(f"- {TABLES_DIR / 'deeplabv3_pretrained_val_all.csv'} rows={len(deeplab_pre_all)}")
    print(f"- {TABLES_DIR / 'deeplabv3_finetuned_best_val_all.csv'} rows={len(deeplab_ft_all)}")
else:
    # When not rerunning, confirm the saved full-validation CSVs exist.
    assert artifact_exists(TABLES_DIR / 'mobilesam_val_all.csv')
    assert artifact_exists(TABLES_DIR / 'fastsam_val_all.csv')
    assert artifact_exists(TABLES_DIR / 'deeplabv3_pretrained_val_all.csv')
    assert artifact_exists(TABLES_DIR / 'deeplabv3_finetuned_best_val_all.csv')

In [ ]:
# Final consistency checks between the raw results and summary CSV files.

summary_1000 = read_csv('final_report_all_model_val_1000_summary.csv')
final_ft = read_csv('deeplabv3_final_finetuning_result.csv')
val_all_dl = read_csv('deeplabv3_val_all_before_after_summary.csv')

# These checks catch accidental mismatch between raw CSVs and final summaries.
pre_1000 = summary_1000[summary_1000['model'].eq('DeepLabV3_pretrained')].iloc[0]
ft_1000 = summary_1000[summary_1000['model'].str.contains('finetuned', regex=False)].iloc[0]
assert np.isclose(final_ft.iloc[0].mean_iou, pre_1000.mean_iou)
assert np.isclose(final_ft.iloc[1].mean_iou, ft_1000.mean_iou)
assert np.isclose(final_ft.iloc[1].delta_vs_pretrained, ft_1000.mean_iou - pre_1000.mean_iou)

pre_all = val_all_dl[val_all_dl['model'].eq('DeepLabV3_pretrained')].iloc[0]
ft_all = val_all_dl[val_all_dl['model'].str.contains('finetuned', regex=False)].iloc[0]
assert np.isclose(ft_all.delta_mean_iou_vs_pretrained, ft_all.mean_iou - pre_all.mean_iou)
assert np.isclose(ft_all.delta_median_iou_vs_pretrained, ft_all.median_iou - pre_all.median_iou)
print('DeepLabV3 before/after summary checks passed')